# Assignment 23: Advanced Intrusion Detection System (IDS)

**Objective**: Develop a robust binary classifier (Normal vs Attack) for the NSL-KDD dataset.

**Improvements over basic implementation**:
- **Data Caching**: Downloads and saves data locally to avoid redundant requests.
- **Feature Selection**: Automatically removes zero-variance features.
- **Hyperparameter Tuning**: Uses `GridSearchCV` to optimize Decision Tree complexity.
- **Optimization**: Uses Early Stopping for the Neural Network to prevent overfitting.
- **Comparison**: Visualizes ROC Curves to compare model performance beyond just accuracy.

In [ ]:
import numpy as np
import pandas as pd
import requests
import os
from io import StringIO

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold

from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import matplotlib.pyplot as plt

# Set style for better plots
plt.style.use('ggplot')
print("Advanced libraries imported.")

In [ ]:
# --- 1. Robust Data Loading with Caching ---

files = {
    "train": ("KDDTrain+.txt", "https://raw.githubusercontent.com/Jehuty4949/NSL_KDD/master/KDDTrain%2B.txt"),
    "test":  ("KDDTest+.txt",  "https://raw.githubusercontent.com/Jehuty4949/NSL_KDD/master/KDDTest%2B.txt")
}

def load_data(name, url):
    if not os.path.exists(name):
        print(f"Downloading {name}...")
        try:
            resp = requests.get(url, timeout=60)
            with open(name, "w") as f:
                f.write(resp.text)
        except Exception as e:
            print(f"Failed to download {name}: {e}")
            return None
    else:
        print(f"Loading {name} from local cache.")
    
    # NSL-KDD column names
    cols = [
        "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
        "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
        "root_shell","su_attempted","num_root","num_file_creations","num_shells",
        "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
        "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
        "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
        "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
        "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
        "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
        "label","difficulty"
    ]
    return pd.read_csv(name, header=None, names=cols)

train_df = load_data(*files["train"])
test_df  = load_data(*files["test"])

# Merge for preprocessing consistency
df = pd.concat([train_df, test_df], ignore_index=True)

# Target: Normal (0) vs Attack (1)
y = (df["label"] != "normal").astype(int)
X = df.drop(columns=["label", "difficulty"])

print(f"Combined Dataset Shape: {df.shape}")
print(f"Class Balance:\n{y.value_counts(normalize=True)}")

In [ ]:
# --- 2. Intelligent Preprocessing ---

# Identify categorical features
cat_cols = ["protocol_type", "service", "flag"]
num_cols = [c for c in X.columns if c not in cat_cols]

# 1. Handle Missing Values (Median for numbers)
# 2. Remove Zero-Variance Features (useless columns like num_outbound_cmds)
# 3. Scale Features (StandardScaler for ANN/NB)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('var_thresh', VarianceThreshold(threshold=0.0)),  # Remove constant columns
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print("Data split into Train and Test sets.")

In [ ]:
# --- 3. Model 1: Gaussian Naive Bayes ---
print("Training Gaussian NB...")
pipe_nb = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GaussianNB())
])
pipe_nb.fit(X_train, y_train)
y_pred_nb = pipe_nb.predict(X_test)
y_proba_nb = pipe_nb.predict_proba(X_test)[:, 1]

acc_nb = accuracy_score(y_test, y_pred_nb)
print(f"Naive Bayes Accuracy: {acc_nb:.4f}")

In [ ]:
# --- 4. Model 2: Decision Tree with Grid Search ---
print("Training Decision Tree (with Simple Grid Search)...")

# We use a separate pipeline for Tree (Scaling is explicitly KEPT here for simplicity of comparison pipeline logic,
# though technically trees don't NEED it. But VarianceThreshold is useful.)
pipe_dt = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

# Grid Search for optimal depth
param_grid = {
    'classifier__max_depth': [10, 20, None],
    'classifier__criterion': ['gini', 'entropy']
}

grid_dt = GridSearchCV(pipe_dt, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_dt.fit(X_train, y_train)

best_dt = grid_dt.best_estimator_
y_pred_dt = best_dt.predict(X_test)
y_proba_dt = best_dt.predict_proba(X_test)[:, 1]

print(f"Best DT Params: {grid_dt.best_params_}")
print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")

In [ ]:
# --- 5. Model 3: MLP (ANN) with Early Stopping ---
print("Training MLP (Neural Network) with Early Stopping...")
pipe_mlp = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', MLPClassifier(
        hidden_layer_sizes=(100, 50), 
        activation='relu', 
        solver='adam', 
        early_stopping=True, 
        validation_fraction=0.1,
        max_iter=50,
        random_state=42
    ))
])

pipe_mlp.fit(X_train, y_train)
y_pred_mlp = pipe_mlp.predict(X_test)
y_proba_mlp = pipe_mlp.predict_proba(X_test)[:, 1]

print(f"MLP Accuracy: {accuracy_score(y_test, y_pred_mlp):.4f}")

In [ ]:
# --- 6. Comparative Analysis & Visualization ---

# 1. Accuracy Comparison
models = ['Naive Bayes', 'Decision Tree', 'MLP']
accuracies = [
    accuracy_score(y_test, y_pred_nb),
    accuracy_score(y_test, y_pred_dt),
    accuracy_score(y_test, y_pred_mlp)
]

plt.figure(figsize=(8, 5))
plt.bar(models, accuracies, color=['skyblue', 'lightgreen', 'orange'])
plt.ylim(0.8, 1.0)
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.005, f"{v:.4f}", ha='center')
plt.show()

# 2. ROC Curves
plt.figure(figsize=(10, 6))

# Helper to plot ROC
def plot_roc(y_true, y_probs, label):
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{label} (AUC = {roc_auc:.3f})')

plot_roc(y_test, y_proba_nb, "Naive Bayes")
plot_roc(y_test, y_proba_dt, "Decision Tree")
plot_roc(y_test, y_proba_mlp, "MLP")

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

# 3. Confusion Matrix for Best Model (likely DT or MLP)
print("Confusion Matrix for Decision Tree (Best Model):")
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_dt, display_labels=["Normal", "Attack"], cmap="Greens")
plt.grid(False)
plt.show()